In [1]:
import pandas as pd
data=pd.read_csv("TATACOFFEE13_21.csv")
data

,Date,Open,High,Low,Close
0,2013-01-01,1410.60,1427.90,1408.30,1415.10
1,2013-01-02,1421.00,1626.60,1416.15,1607.40
2,2013-01-03,1632.55,1673.90,1613.05,1626.20
3,2013-01-04,1627.75,1627.75,1574.60,1579.05
4,2013-01-07,1580.00,1639.50,1565.50,1595.65
...,...,...,...,...,...
2220,2021-12-22,202.90,207.80,201.35,205.00
2221,2021-12-23,206.00,206.85,202.05,202.95
2222,2021-12-24,203.90,203.90,199.35,201.00
2223,2021-12-27,200.00,222.00,196.00,218.35


In [2]:
column = "Close"

from sklearn.preprocessing import MinMaxScaler

Ms = MinMaxScaler()

data1 = Ms.fit_transform(data[[column]])

print("Len:", data1.shape)

Len: (2225, 1)


In [3]:
training_size = round(len(data1) * 0.95)

print(training_size)

X_train = data1[:training_size]
X_test = data1[training_size:]

print("X_train length:", X_train.shape)
print("X_test length:", X_test.shape)

y_train = data1[:training_size]
y_test = data1[training_size:]

print("y_train length:", y_train.shape)
print("y_test length:", y_test.shape)

2114
X_train length: (2114, 1)
X_test length: (111, 1)
y_train length: (2114, 1)
y_test length: (111, 1)


In [5]:
performance = {
    "Model": [],
    "RMSE": [],
    "Mape": [],
    "Lag": [],
    "Test": []
}

In [17]:
def combination(data, listt):

    print(listt)

    datasettwo = data[listt]

    test_obs = 28

    train = datasettwo[:-test_obs]
    test = datasettwo[-test_obs:]

    from statsmodels.tsa.api import VAR

    for i in range(1, 11):

        model = VAR(train)

        results = model.fit(i)

        print("Order :", i)
        print("AIC :", results.aic)
        print("BIC :", results.bic)
        print()

    
    x = model.select_order(maxlags=12)

    order = x.selected_orders["aic"]

    print("Selected Order:", order)

    result = model.fit(order)
    lagged_values = train.values[-order:]


    pred = result.forecast(
        y=lagged_values,
        steps=28
    )

    pred = pd.DataFrame(
        pred,
        columns=listt
    )
    pred.to_csv(
        "predicted_forecast.csv",
        index=False
    )

    # Calculate RMSE and MAPE
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_percentage_error

    mse = mean_squared_error(
        test,
        pred
    )

    rmse = mse ** 0.5

    mape = mean_absolute_percentage_error(
        test,
        pred
    )

    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["Mape"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test)

    perf = pd.DataFrame(performance)

    return result, perf

In [7]:
listt = ["Close", "High", "Open", "Low"]

listt2 = [
    "Adj Close",
    "Volume",
    "PM2.5",
    "NO2",
    "NO",
    "NH3",
    "SO2",
    "CO",
    "year"
]

In [18]:
result, perf = combination(data, listt)

['Close', 'High', 'Open', 'Low']
Order : 1
AIC : 17.468056619203143
BIC : 17.519918120137838

Order : 2
AIC : 17.14473236309155
BIC : 17.23811812332769

Order : 3
AIC : 17.147708041702877
BIC : 17.282649265642455

Order : 4
AIC : 17.148827299213142
BIC : 17.325355229396642

Order : 5
AIC : 17.11977250561703
BIC : 17.337918422787514

Order : 6
AIC : 17.03761957361099
BIC : 17.297414796778288

Order : 7
AIC : 16.98134053047922
BIC : 17.282816416984254

Order : 8
AIC : 16.98547728645597
BIC : 17.328665232035238

Order : 9
AIC : 16.969321380806726
BIC : 17.354252819656907

Order : 10
AIC : 16.948085953037864
BIC : 17.374792357880565

Selected Order: 11


In [19]:
perf

,Model,RMSE,Mape,Lag,Test
0,"[Close, High, Open, Low]",15.299683,0.066959,11,Close High Open Low 2197 225...
1,"[Close, High, Open, Low]",15.299683,0.066959,11,Close High Open Low 2197 225...
2,"[Close, High, Open, Low]",15.299683,0.066959,11,Close High Open Low 2197 225...
